# [3.1] Chỉ chạy SigLIP - Tải Trực Tiếp Từ Modal (Batch 2) → Volume `keyframes-btc`

`docs/triet/ke-hoach-offline-ingest.md` §2.1, §2.3, §7 · config `config/ingest_shards_batch2_btc_images.yaml`

Chạy **sau** `02-keyframes-btc-import.ipynb`. Cùng bundle, cùng volume, **cùng cách chia người** —
đổi `MEMBER` ở ô đầu là ra đúng phần việc bạn đã nhập ở lượt trước.

### Ba tầng, và tầng nào KHÔNG có ở đây

| Tầng | Model | Ra artifact |
|---|---|---|
| `n4_siglip` | `google/siglip-so400m-patch14-384` (tháp ảnh) | `siglip_vectors` |
| `n5_ocr_easyocr` | easyocr vi+en | `ocr_text` |
| `n5_ocr_qwen` | `Qwen/Qwen2-VL-7B-Instruct` 4-bit | `sviptr_ocr` |

`n4_objects` **không có**: BTC phát hành sẵn nhãn object và notebook 02 đã nạp chúng. Chạy
detector của ta lên chính bộ keyframe này là phép đo khác (§3.6) và là việc khác.

`n6_recap` / `n6_bge_m3` **không có**: Lead đã hoãn tầng ReCap.

### Vì sao ảnh tải lại từ zip chứ không kéo từ Volume

Volume ở đường BTC là **ghi một chiều**. Kéo ảnh về là ~177.000 lần `read_file`; tải lại gói
`Keyframes_*.zip` là 1,4–5,7 GB một lần, có resume. Cùng dữ liệu, khác hai bậc chi phí.

### Ba model cùng nằm trong VRAM một lúc — có chủ ý

SigLIP ~1,1 GB + easyocr ~0,1 GB + Qwen2-VL 4-bit ~5,5 GB ≈ **6,7 GB / 15 GB của T4**. Giữ cả ba
để mỗi model chỉ nạp **một lần cho cả phiên**; nạp lại theo từng video tốn **5–16 giờ mỗi người
chỉ để nạp** (đo trên số video thật của từng người).

> Nếu gặp OOM: đặt `KEEP_MODELS = False` ở ô đầu. Notebook sẽ thả model sau mỗi video — đúng hành
> vi cũ của §4.2, đổi lại là thời gian nạp.

### `SKIP_STAGES` — bỏ qua một tầng cho LƯỢT CHẠY NÀY

Mặc định đang là `["n5_ocr_qwen"]`: chạy SigLIP + easyocr trước, để Qwen lại sau. Muốn chỉ chạy
SigLIP thì thêm `"n5_ocr_easyocr"` vào danh sách.

Bỏ qua ở đây **không cắm marker**, nên lần chạy sau vẫn thấy tầng đó là việc còn lại và chạy tiếp
từ chỗ đứt. Đó là khác biệt so với việc **xoá tầng khỏi yaml**:

| | `SKIP_STAGES` | xoá khỏi `config/…_btc_images.yaml` |
|---|---|---|
| lượt này | bỏ qua | bỏ qua |
| lần chạy sau | vẫn là việc còn lại | biến mất khỏi kế hoạch |
| `merge_bundle.py` | **vẫn đòi đủ mảnh** | không đòi nữa |

Nên chừng nào `n5_ocr_qwen` còn trong yaml thì **`merge_bundle.py` sẽ từ chối bundle** cho tới khi
nó chạy xong cả 873 video. Đó đúng là điều ta muốn lúc này (hoãn, không bỏ). Nếu Lead quyết **bỏ
hẳn** Qwen — có cơ sở: đo được 10,8 giây/ảnh, tức ~532 giờ GPU cho batch 1, và lượt 4 đã sinh
`paddle_ocr` + `viet_ocr` thay chỗ — thì phải xoá tầng khỏi yaml, không phải để nó ở `SKIP_STAGES`.

> ⚠️ Ô Step 0b từng có dòng `MY_VIDEOS = MY_VIDEOS[:1]` — di sản của một lần chạy thử, nó cắt phần
> việc của mọi người xuống **một video**. Đã xoá.


## Dùng lại private Kaggle cache

Attach cùng Dataset đã dùng cho notebook 02. Kaggle thường tự giải nén archive thành thư mục;
notebook nhận cả layout đã giải nén lẫn ZIP nguyên bản trong `/kaggle/input`, giữ cache read-only
và chỉ tải từ website khi cả hai dạng đều thiếu. Không copy 15 GiB sang `/kaggle/working`.


## Step 0a — repo + gói

In [ ]:
import re
MEMBER        = "bao1"         # <-- ĐỔI THÀNH TÊN BẠN (VD: bao1, bao2..., bao16)
TOTAL_WORKERS = 16             # Đã chốt 16 máy Kaggle chạy đồng thời
# Tự động trích xuất số từ tên (bao1 -> index 0, bao16 -> index 15)
WORKER_INDEX  = int(re.search(r'\d+', MEMBER).group()) - 1

TRANSPORT   = "modal"
VOLUME      = "keyframes-batch-2-v2"  # CÙNG volume với notebook 02
VOL_VERSION = 2
KEEP_MODELS = True             # False = thả model sau mỗi video

ONLY_VIDEOS = []
REDO_STAGES = []
REPO_URL    = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF    = "feature/triet/structure"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/input/mfe-repo")
if not REPO.is_dir():
    REPO = Path("/kaggle/working/mfe")
    if not REPO.is_dir():
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF, url, str(REPO)],
                       check=True, capture_output=True)
sys.path.insert(0, str(REPO / "src"))

req = REPO / "notebooks" / "requirements-kaggle.lock.txt"
if not req.is_file():
    req = REPO / "notebooks" / "requirements-kaggle-btc-images.txt"
print(f"cài từ {req.name}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)


## Step 0b — Modal, phân công, việc đã xong

In [ ]:
# download hàm giả vì không còn dùng tới
def download(*args, **kwargs): pass
MAPS_DIR = None
print(f"đĩa trống: {free_gb():.1f} GB")

## Step 1 — vòng lặp

Theo **video**, không theo tầng: model đã nằm trong cache nên số lần nạp giống hệt cách chia theo
tầng, nhưng mỗi video chỉ cần ~35 MB ảnh trên đĩa thay vì cả gói đã bung.

Marker cắm theo từng *(tầng, video)*. Phiên đứt giữa chừng thì lần chạy sau bỏ đúng những việc đã
xong, kể cả khi nó đứt giữa hai tầng của cùng một video.

`merge=True` khi ghi manifest: bundle này được đổ bởi **ba notebook** (02 nhập, 03 đây, 04 video)
và cả ba cùng ghi `part-manifest/{video}.json`. Không gộp thì mỗi lượt xoá sạch artifact của lượt
trước, còn marker vẫn báo "xong" nên không lần chạy nào phục hồi lại.

In [ ]:
import time, traceback
import modal

def stages_for(video_id):
    return [s for s in STAGE_LIST
            if s in REDO_STAGES or not status.is_done(s, video_id)]

failures = []
started = time.perf_counter()
done_count = 0

def run_one(video_id, kf_dir, map_csv, stages):
    keyframes = read_keyframes(map_csv, kf_dir)
    ctx = StageContext(video_id, Path("/dev/null"), keyframes, writer, layout, WORK)
    markers = []
    for stage in stages:
        ctx.params = dict(PARAMS.get(stage, {}))
        marker = STAGES[stage](ctx)
        markers.append(marker)
        status.done.add((stage, video_id))
        print(f"      {stage:<16} {marker.get('rows', '-'):>6} dòng  {marker.get('elapsed_s')}s")
    return keyframes, markers

print("Kết nối Modal Volume để kéo ảnh trực tiếp...")
vol = modal.Volume.from_name(VOLUME)

mine = set(MY_VIDEOS)
if ONLY_VIDEOS:
    mine &= set(ONLY_VIDEOS)
    
# Tìm các video cần chạy
pending = [v for v in mine if status.is_done("n0_import_btc", v) and stages_for(v)]

print(f"\n=== Tổng cộng: còn {len(pending)} video cần chạy SigLIP ===")

for video_id in pending:
    stages = stages_for(video_id)
    try:
        remote_kf_dir = f"bundle/{BUNDLE_NAME}/keyframes/{video_id}"
        kf_dir = WORK / "keyframes" / video_id
        
        # Tải thẳng ảnh giải nén sẵn từ Modal về thư mục tạm
        _, n_img = download_modal_dir(vol, remote_kf_dir, kf_dir)
        
        # Tải file CSV từ Modal (Cực kỳ quan trọng để SigLIP đọc map khung hình)
        remote_csv = f"bundle/{BUNDLE_NAME}/map-keyframes/{video_id}.csv"
        map_csv = WORK / "map-keyframes" / f"{video_id}.csv"
        map_csv.parent.mkdir(parents=True, exist_ok=True)
        with map_csv.open("wb") as f:
            for chunk in vol.read_file(remote_csv):
                f.write(chunk)
                                   
        print(f"  {video_id}  {n_img} ảnh  ({len(stages)} tầng)")
        keyframes, markers = run_one(video_id, kf_dir, map_csv, stages)
        
        # BundleWriter TỰ ĐỘNG đẩy output lên Modal (ví dụ: bundle/aic_btc_b2_v1/feature/triet/siglip_vectors/...)
        writer.write_part_manifest(video_id, markers, keyframes=len(keyframes), merge=True)
        
        done_count += 1
        print(f"  [{done_count:>3}/{len(pending)}] {video_id} xong")
    except Exception as err:
        failures.append((video_id, repr(err)))
        print(f"  [LỖI] {video_id}: {err}")
        traceback.print_exc(limit=3)
    finally:
        shutil.rmtree(WORK / "keyframes" / video_id, ignore_errors=True)
        (WORK / "map-keyframes" / f"{video_id}.csv").unlink(missing_ok=True)
        if not KEEP_MODELS:
            unload_models()

print("\n" + "=" * 72)
print(f"phiên này: {done_count} video, {len(failures)} lỗi, "
      f"{(time.perf_counter() - started) / 3600:.1f}h")
for video_id, err in failures:
    print(f"  {video_id}: {err}")


## Step 2 — chốt phiên

In [ ]:
unload_models()

fresh = StageStatus.load(transport, layout, MEMBER)
left  = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not fresh.is_done(s, v)]

total = len(MY_VIDEOS) * len(STAGE_LIST)
print(f"{MEMBER}: {total - len(left)}/{total} việc đã lên {VOLUME}")
for stage in STAGE_LIST:
    n = sum(1 for v in MY_VIDEOS if fresh.is_done(stage, v))
    print(f"  {stage:<16} {n:>4}/{len(MY_VIDEOS)} video")

if left:
    print()
    print(f"còn {len(left)} việc — bấm Run All lại, nó chạy tiếp từ chỗ đứt")
else:
    print()
    print("XONG phần của bạn. Báo Lead.")
    print(f"  python scripts/verify_bundle.py --shards {SHARDS} --member {MEMBER}")